<a href="https://colab.research.google.com/github/rachnashetty5/hinglish-and-nyishi-english-translation/blob/main/Nyishi_English_Translator_T5_Projectipynb.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import torch
torch.cuda.is_available()

True

In [2]:
!pip install pandas openpyxl transformers datasets sentencepiece accelerate

In [3]:
import pandas as pd

file_path = "/content/English-Nyishi Training Data 2025.xlsx"

df = pd.read_excel(file_path)

df.head()

,English,Nyishi
0,weather beaten,og layil tixum tiqhiqnam
1,weather man,doonyi poolam betambo nyi
2,weather vane,dooi yinam mam kaan nan
3,well done,alwb nyipa
4,well groomed,dvrwq dvb vj koonam


In [4]:
df.columns

Index(['English', 'Nyishi'], dtype='object')

In [5]:
df = df.rename(columns={
    "Nyishi": "input_text",
    "English": "target_text"
})

df.head()

,target_text,input_text
0,weather beaten,og layil tixum tiqhiqnam
1,weather man,doonyi poolam betambo nyi
2,weather vane,dooi yinam mam kaan nan
3,well done,alwb nyipa
4,well groomed,dvrwq dvb vj koonam


In [6]:
df["input_text"] = "translate Nyishi to English: " + df["input_text"]

In [7]:
df["input_text"] = df["input_text"].astype(str)
df["target_text"] = df["target_text"].astype(str)

In [8]:
df = df.dropna()
df = df[df["input_text"].str.strip() != ""]
df = df[df["target_text"].str.strip() != ""]

In [9]:
from datasets import Dataset

dataset = Dataset.from_pandas(df)

dataset = dataset.train_test_split(test_size=0.1)

dataset

DatasetDict({
    train: Dataset({
        features: ['target_text', 'input_text'],
        num_rows: 54000
    })
    test: Dataset({
        features: ['target_text', 'input_text'],
        num_rows: 6000
    })
})

In [10]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

model_name = "google/mt5-small"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Loading weights:   0%|          | 0/192 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to encoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie shared.weight to decoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


In [11]:
def tokenize(example):

    model_inputs = tokenizer(
        example["input_text"],
        max_length=64,
        padding="max_length",
        truncation=True
    )

    labels = tokenizer(
        text_target=example["target_text"],
        max_length=64,
        padding="max_length",
        truncation=True
    )

    model_inputs["labels"] = labels["input_ids"]

    return model_inputs


tokenized_dataset = dataset.map(tokenize)

Map:   0%|          | 0/54000 [00:00<?, ? examples/s]

Map:   0%|          | 0/6000 [00:00<?, ? examples/s]

In [12]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="./t5_nyishi_model",
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=3,
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=5e-5
)

In [13]:
from transformers import Trainer

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset["train"],
    eval_dataset=tokenized_dataset["test"]
)

trainer.train()

Epoch,Training Loss,Validation Loss
1,0.489987,0.406993
2,0.418710,0.358663
3,0.403115,0.345983


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=20250, training_loss=1.1022485061457128, metrics={'train_runtime': 7128.9853, 'train_samples_per_second': 22.724, 'train_steps_per_second': 2.841, 'total_flos': 1.070718124032e+16, 'train_loss': 1.1022485061457128, 'epoch': 3.0})

In [14]:
def translate_nyishi_to_english(text):

    input_text = "translate Nyishi to English: " + text

    inputs = tokenizer(input_text, return_tensors="pt").to(model.device)

    outputs = model.generate(
        **inputs,
        max_length=64,
        num_beams=5
    )

    return tokenizer.decode(outputs[0], skip_special_tokens=True)

In [20]:
translate_nyishi_to_english("Air pollution ami health lo problem piinyi do.")

"It's a very dangerous problem"